In [1]:
from pathlib import Path
import sys
import json
from datetime import datetime

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Llm.llm_loader import LLM
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv(project_root / '.env')

True

In [2]:
def load_data(project_root, dataset_name):
    datasets = [
        (
            'two_table',
            project_root / 'Data' / dataset_name / 'Synthesized_two_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset_name / 'Synthesized_two_table_with_spider_db_id.json')
        ),
        (
            'three_table',
            project_root / 'Data' / dataset_name / 'Synthesized_three_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset_name / 'Synthesized_three_table_with_spider_db_id.json')
        ),
    ]
    return datasets

In [3]:
def append_log_entry(log_records, dataset_name, data_split, row, response_text,Answer_llm,log_path):
    log_records.append(
        {
            'model': Answer_llm.model_name,
            'provider': Answer_llm.provider,
            'id': f"{dataset_name}-{data_split}-{row['id_']}",
            'spider_db_id': row['Spider_db_id'],
            'question': row['Question'],
            'response': response_text,
        }
    )
    log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')

In [4]:
def run_(project_root,prompt_path, log_path, datasets,dataset_name,Answer_llm):
    prompt_template = prompt_path.read_text(encoding='utf-8').strip()
    log_records = []
    log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')
    for data_split, data_source, df in datasets:
        for _, row in tqdm(df.iterrows(), total=len(df), desc=data_split):
            schema_path = f"{project_root}/Data/MMQA/schema_descriptions/{row['Spider_db_id']}.csv"
            schema_df = pd.read_csv(schema_path)
            database_schema = schema_df.to_markdown()
            prompt = (
                prompt_template
                .replace('{DATABASE_SCHEMA}', database_schema)
                .replace('{QUESTION}', row['Question'])
                .replace('{HINT}', 'No hint')
            )
            res = Answer_llm.query(prompt)
            append_log_entry(log_records, dataset_name, data_split, row, res,Answer_llm,log_path)

    print(f'All responses have been saved to {log_path}')

In [5]:
dataset_name = 'MMQA'
datasets = load_data(project_root, dataset_name)

ollama_llm_lists = ['gemma3:27b']
gpt_llm_lists = ['gpt-3.5-turbo', 'gpt-4o-mini']
# gpt_llm_lists = ['gpt-3.5-turbo', 'gpt-4o-mini','gpt-5-mini'] 
# gpt-3.5-turbo-0125, gpt-4o-mini-2024-07-18, gpt-5-mini-2025-08-07
# Intelligence: 1, 2, 3
# Speed: 2, 4, 4
# Reasoning tokens: no, no, yes
# Knowledge Cutoff: Sep 01, 2021; Oct 01, 2023; May 31, 2024

In [6]:
method_ = 'few_shot'
prompt_path = project_root / 'Templates' / method_ / 'baseline_schema_linking.txt'

In [7]:

for model_name in ollama_llm_lists:
    Answer_llm = LLM(model_name = model_name, provider= 'ollama')
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    logs_dir = project_root / 'Logs' / model_name
    logs_dir.mkdir(exist_ok=True)
    log_path = logs_dir / f'{method_}_baseline_schema_linking_{dataset_name}_{run_id}.json'
    run_(project_root,prompt_path, log_path, datasets,dataset_name,Answer_llm)

three_table: 100%|██████████| 721/721 [24:42<00:00,  2.06s/it]

All responses have been saved to /home/xubeiyu/projects/Schemalinking/Logs/gemma3:27b/few_shot_baseline_schema_linking_MMQA_20260311_150919.json


In [ ]:
for model_name in gpt_llm_lists:
    Answer_llm = LLM(model_name = model_name, provider= 'openai')
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    logs_dir = project_root / 'Logs' / model_name
    logs_dir.mkdir(exist_ok=True)
    log_path = logs_dir / f'{method_}_baseline_schema_linking_{dataset_name}_{run_id}.json'
    run_(project_root,prompt_path, log_path, datasets,dataset_name,Answer_llm)